In [45]:
import pandas as pd
import math

In [3]:
ratings = pd.read_csv(
    "../data/ml-1m/ratings.dat",
    sep="::",
    names=["user_id", "movie_id", "rating", "timestamp"],
    engine="python",
)
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [4]:
ratings.shape

(1000209, 4)

In [5]:
ratings["user_id"].nunique()

6040

In [7]:
n_users = ratings["user_id"].nunique()
n_movies = ratings["movie_id"].nunique()
n_ratings = len(ratings)

density = n_ratings / (n_users * n_movies)
print(density)

0.044683625622312845


In [12]:
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
print(ratings["datetime"].min())
print(ratings["datetime"].max())

2000-04-25 23:05:32
2003-02-28 17:49:50


In [24]:
ratings_sorted = ratings.sort_values("timestamp").reset_index(drop=True)
cutoff = int(len(ratings_sorted) * 0.8)
train = ratings_sorted[:cutoff]
test = ratings_sorted[cutoff:]
print(len(train))
print(len(test))
print(len(ratings_sorted))

800167
200042
1000209


In [34]:
train_users = set(train["user_id"])
test_users = set(test["user_id"])
print(len(test_users))
print(len(test_users - train_users))
print(len(test_users - train_users) / len(test_users)) 

1783
640
0.35894559730790804


In [35]:
test_warm = test[test["user_id"].isin(train_users)]
test["user_id"].nunique() - test_warm["user_id"].nunique()

640

In [ ]:
def precision_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    """Compute Precision@k for a single user's recommendations.

    Of the top-k recommended items, what fraction were actually relevant.

    Args:
        recommended: Movie IDs in ranked order (best first).
        relevant: Movie IDs the user actually liked (rating >= 4).
        k: Number of top recommendations to evaluate.

    Returns:
        Fraction of the top-k recommendations that were relevant, in [0, 1].
    """
    top_k = recommended[:k]
    hits = len([movie for movie in top_k if movie in relevant])
    return hits / k

In [43]:
def recall_at_k(recommended: list[int], relevant: set[int], k: int) -> float | None:
    """Compute Recall@k for a single user's recommendations.

    Of all the items the user actually liked, what fraction appeared
    in the top-k recommendations.

    Args:
        recommended: Movie IDs in ranked order (best first).
        relevant: Movie IDs the user actually liked (rating >= 4).
        k: Number of top recommendations to evaluate.

    Returns:
        Fraction of relevant items captured in the top-k, in [0, 1],
        or None if the user has no relevant items (recall is undefined).
    """
    if not relevant:
        return None
    top_k = recommended[:k]
    hits = len([movie for movie in top_k if movie in relevant])
    return hits / len(relevant)

In [46]:
def ndcg_at_k(recommended: list[int], relevant: set[int], k: int) -> float | None:
    """Compute NDCG@k for a single user's recommendations.

    Rewards placing relevant items higher in the ranking via a logarithmic
    position discount, then normalizes by the ideal ranking so scores are
    comparable across users.

    Args:
        recommended: Movie IDs in ranked order (best first).
        relevant: Movie IDs the user actually liked (rating >= 4).
        k: Number of top recommendations to evaluate.

    Returns:
        NDCG in [0, 1], where 1.0 is a perfect ranking, or None if the
        user has no relevant items (NDCG is undefined).
    """
    if not relevant:
        return None

    top_k = recommended[:k]

    # DCG: walk the ranked list, each relevant hit contributes a
    # discounted gain based on its position (1-indexed).
    dcg = 0.0
    for i, movie in enumerate(top_k):
        position = i + 1  # enumerate starts at 0; positions start at 1
        if movie in relevant:
            dcg += 1 / math.log2(position + 1)

    # IDCG: the best possible DCG — all relevant items packed at the top.
    # The number of ideal "hits" is limited by both how many relevant
    # items exist and how many slots (k) we have.
    ideal_hits = min(len(relevant), k)
    idcg = 0.0
    for i in range(ideal_hits):
        position = i + 1
        idcg += 1 / math.log2(position + 1)

    return dcg / idcg